In [45]:
import pandas as pd
import numpy as np
import kaiwu as kw
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import MDS
import matplotlib.patches as mpatches
# Seaborn主题
sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = ['SimSun'] 
plt.rcParams['axes.unicode_minus'] = False

# 问题一

> 读取参考算例.xlsx的第二个表单，设置为旅行时间矩阵

In [39]:
# 读取第二个表单，跳过第一行
excel_path = 'v1/参考算例.xlsx'
xl = pd.ExcelFile(excel_path)
sheet_names = xl.sheet_names
target_sheet = sheet_names[1]
df = pd.read_excel(excel_path, sheet_name=target_sheet, header=0)
# 转换为numpy数组
travel_time_matrix = df.iloc[:, 1:].values
travel_time_matrix

array([[0, 2, 2, ..., 4, 6, 2],
       [2, 0, 4, ..., 4, 5, 1],
       [2, 4, 0, ..., 5, 8, 4],
       ...,
       [4, 4, 5, ..., 0, 2, 4],
       [6, 5, 8, ..., 2, 0, 6],
       [2, 1, 4, ..., 4, 6, 0]], shape=(51, 51))

In [40]:
w = travel_time_matrix
n = w.shape[0]
x = kw.core.ndarray((n, n), "x", kw.core.Binary)
edges = [(u, v) for u in range(n) for v in range(n) if w[u, v] != 0]
no_edges = [(u, v) for u in range(n) for v in range(n) if w[u, v] == 0]

def is_edge_used(x, u, v):
    return kw.core.quicksum([x[u, j] * x[v, j + 1] for j in range(-1, n - 1)])

penalty = 10.0
qubo_model = kw.qubo.QuboModel()
# TSP时间成本目标函数
qubo_model.set_objective(kw.core.quicksum([w[u, v] * is_edge_used(x, u, v) for u, v in edges]))

# 约束1：每个位置只能有一个节点
qubo_model.add_constraint((x.sum(axis=0) - 1) ** 2 == 0, "sequence_cons", penalty=penalty)

# 约束2：每个节点只能出现在一个位置
qubo_model.add_constraint((x.sum(axis=1) - 1) ** 2 == 0, "node_cons", penalty=penalty)

# 约束3：禁止使用不存在的边
qubo_model.add_constraint(kw.core.quicksum([is_edge_used(x, u, v) for u, v in no_edges]),
    "connect_cons", penalty=penalty)

In [41]:
# 运行模拟退火算法求解QUBO模型
solver = kw.solver.SimpleSolver(kw.classical.SimulatedAnnealingOptimizer(initial_temperature=100,
                                                                         alpha=0.99,
                                                                         cutoff_temperature=0.001,
                                                                         iterations_per_t=100,
                                                                         size_limit=100))

sol_dict, qubo_val = solver.solve_qubo(qubo_model)

In [42]:
# 检查约束是否满足
unsatisfied_count, res_dict = qubo_model.verify_constraint(sol_dict)

# 计算时间成本
path_val = kw.core.get_val(qubo_model.objective, sol_dict)

if unsatisfied_count == 0:
    # 获取矩阵值
    x_val = kw.core.get_array_val(x, sol_dict)
    # 获取非零元素的索引
    nonzero_index = np.array(np.nonzero(x_val.T))[1]
else:
    print('invalid path')

# 把哈密顿回路平移了！
idx = np.where(nonzero_index == 0)[0][0]
rotated_path = np.concatenate([nonzero_index[idx:], nonzero_index[:idx]])
complete_path = np.concatenate([rotated_path, [0]])
print('时间成本:{}'.format(path_val))
print("未满足的约束数量:", unsatisfied_count)
print("约束项的值:", res_dict)
print('完整路径: {}'.format(complete_path))

时间成本:231.0
未满足的约束数量: 0
约束项的值: {'constraint0': np.float64(0.0), 'constraint2': np.float64(0.0), 'constraint4': np.float64(0.0), 'constraint6': np.float64(0.0), 'constraint8': np.float64(0.0), 'constraint10': np.float64(0.0), 'constraint12': np.float64(0.0), 'constraint14': np.float64(0.0), 'constraint16': np.float64(0.0), 'constraint18': np.float64(0.0), 'constraint20': np.float64(0.0), 'constraint22': np.float64(0.0), 'constraint24': np.float64(0.0), 'constraint26': np.float64(0.0), 'constraint28': np.float64(0.0), 'constraint30': np.float64(0.0), 'constraint32': np.float64(0.0), 'constraint34': np.float64(0.0), 'constraint36': np.float64(0.0), 'constraint38': np.float64(0.0), 'constraint40': np.float64(0.0), 'constraint42': np.float64(0.0), 'constraint44': np.float64(0.0), 'constraint46': np.float64(0.0), 'constraint48': np.float64(0.0), 'constraint50': np.float64(0.0), 'constraint52': np.float64(0.0), 'constraint54': np.float64(0.0), 'constraint56': np.float64(0.0), 'constraint58': n

> 导出为csv，用于量子计算

In [43]:
qubo_mat = qubo_model.get_matrix()
pd.DataFrame(qubo_mat).to_csv("tsp1.csv", index=False, header=False)

> 哈密顿量演化曲线

> 路径可视化图

## 问题二